<a href="https://colab.research.google.com/github/janpfrang-hash/distance-measurement-sensor/blob/main/poti_travel_readout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# 1. Install the rainflow and ipyfilechooser libraries
!pip install rainflow ipyfilechooser -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rainflow
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import drive
from ipyfilechooser import FileChooser
from scipy import stats  # Added for the degradation proof

# 2. Mount Google Drive
drive.mount('/content/drive')

print("\n--- Fatigue Data Analyzer: Full Spectrum & Degradation Analysis ---")

# 3. Create the UI widgets
fc = FileChooser('/content/drive/MyDrive')
fc.title = '<b>1. Select your Logfile (.txt, .csv)</b>'
fc.filter_pattern = ['*.txt', '*.csv']

# Tuning widgets
mag_threshold_input = widgets.FloatText(
    value=0.20,
    step=0.05,
    description='Min Mag (mm):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

rounding_input = widgets.Dropdown(
    options=[('No Rounding', None), ('2 Decimals (0.01 mm)', 2), ('1 Decimal (0.1 mm)', 1)],
    value=2,
    description='Smoothing:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

# Outlier filtering widgets
filter_outliers_checkbox = widgets.Checkbox(
    value=False,
    description='Remove Outliers >',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px')
)

max_mag_input = widgets.FloatText(
    value=10.0,
    step=0.5,
    layout=widgets.Layout(width='80px')
)

# Force Calibration widgets
force_ref_input = widgets.FloatText(
    value=220.0,
    step=10.0,
    description='Ref Force (N):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

disp_ref_input = widgets.FloatText(
    value=5.8,
    step=0.1,
    description='at Ref Mag (mm):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

process_button = widgets.Button(
    description="Generate Full Analysis",
    button_style='primary',
    icon='bar-chart',
    layout=widgets.Layout(width='300px', height='40px')
)
output = widgets.Output()

ui_box = widgets.VBox([
    fc,
    widgets.HTML("<b>2. Fine-Tune Algorithm (Noise & Outlier Filtering)</b>"),
    widgets.HBox([mag_threshold_input, rounding_input]),
    widgets.HBox([filter_outliers_checkbox, max_mag_input, widgets.Label(" mm")]),
    widgets.HTML("<br><b>3. Force Calibration (for Stiffness & Secondary Axes)</b>"),
    widgets.HBox([force_ref_input, disp_ref_input]),
    widgets.HTML("<br>"),
    process_button
])

display(ui_box, output)

def analyze_fatigue_data(b):
    with output:
        output.clear_output()

        filepath = fc.selected
        min_mag = mag_threshold_input.value
        rounding_decimals = rounding_input.value

        filter_outliers = filter_outliers_checkbox.value
        max_mag = max_mag_input.value

        ref_force = force_ref_input.value
        ref_mag = disp_ref_input.value

        # Calculate static stiffness factor (N/mm) for the histogram axis
        stiffness = ref_force / ref_mag if ref_mag != 0 else 1.0

        if not filepath:
            print("⚠️ Please select a file from the list first!")
            return

        print(f"Loading data from: {filepath}...")

        try:
            # Load BOTH the timestamp and displacement columns
            df = pd.read_csv(filepath, sep=';', usecols=['Zeit_s', 'Weg_mm'])
        except Exception as e:
            print(f"❌ Error loading file. Details: {e}")
            return

        # --- Apply Signal Smoothing (Hysteresis Bypass) ---
        if rounding_decimals is not None:
            print(f"Smoothing signal (rounding to {rounding_decimals} decimal places)...")
            df['Weg_mm'] = df['Weg_mm'].round(rounding_decimals)

        print("Extracting rainflow cycles...")
        cycles = rainflow.extract_cycles(df['Weg_mm'])

        magnitudes = []
        counts = []
        cycle_times = []

        for rng, mean, count, i_start, i_end in cycles:
            mag = rng

            # Apply the user-defined threshold
            if mag >= min_mag:
                # Apply outlier filtering if enabled
                if filter_outliers and mag > max_mag:
                    continue

                magnitudes.append(mag)
                counts.append(count)
                # Map the cycle to the time it finished
                cycle_times.append(df['Zeit_s'].iloc[i_end])

        if not magnitudes:
            print(f"No valid cycles found matching your criteria.")
            return

        print("Processing bins, calculating statistics, and generating plots...")
        magnitudes = np.array(magnitudes)
        counts = np.array(counts)
        cycle_times = np.array(cycle_times)

        # --- Calculate Statistics ---
        expanded_mags = np.repeat(magnitudes, np.ceil(counts).astype(int))

        total_cycles = np.sum(counts)
        avg_mag = np.mean(expanded_mags)
        max_mag_val = np.max(expanded_mags)
        min_mag_val = np.min(expanded_mags)
        std_mag = np.std(expanded_mags)

        avg_minus_2std = avg_mag - (2 * std_mag)
        avg_plus_2std  = avg_mag + (2 * std_mag)
        avg_minus_3std = avg_mag - (3 * std_mag)
        avg_plus_3std  = avg_mag + (3 * std_mag)
        avg_minus_4std = avg_mag - (4 * std_mag)
        avg_plus_4std  = avg_mag + (4 * std_mag)

        stats_df = pd.DataFrame({
            'Statistic': [
                'Total Detected Cycles',
                'Average Magnitude',
                'Absolute Max Magnitude',
                'Absolute Min Magnitude',
                'Standard Deviation',
                'Min (Average - 2 Std Dev)',
                'Max (Average + 2 Std Dev)',
                'Min (Average - 3 Std Dev)',
                'Max (Average + 3 Std Dev)'
            ],
            'Magnitude': [
                f"{total_cycles:,.2f}",
                f"{avg_mag:.2f} mm",
                f"{max_mag_val:.2f} mm",
                f"{min_mag_val:.2f} mm",
                f"{std_mag:.2f} mm",
                f"{avg_minus_2std:.2f} mm",
                f"{avg_plus_2std:.2f} mm",
                f"{avg_minus_3std:.2f} mm",
                f"{avg_plus_3std:.2f} mm"
            ],
            'Calculated Force': [
                "-",
                f"{avg_mag * stiffness:.2f} N",
                f"{max_mag_val * stiffness:.2f} N",
                f"{min_mag_val * stiffness:.2f} N",
                f"{std_mag * stiffness:.2f} N",
                f"{avg_minus_2std * stiffness:.2f} N",
                f"{avg_plus_2std * stiffness:.2f} N",
                f"{avg_minus_3std * stiffness:.2f} N",
                f"{avg_plus_3std * stiffness:.2f} N"
            ]
        })

        # ==========================================
        # PLOT 1: HISTOGRAM
        # ==========================================
        max_mag_bin = magnitudes.max()
        bins = np.arange(min_mag, max_mag_bin + 0.02, 0.02)
        bin_indices = np.digitize(magnitudes, bins)

        binned_counts = np.zeros(len(bins))
        for i, bin_idx in enumerate(bin_indices):
            if bin_idx - 1 < len(bins):
                binned_counts[bin_idx - 1] += counts[i]

        fig, ax1 = plt.subplots(figsize=(14, 6))

        ax1.bar(bins, binned_counts, width=0.018, align='edge', edgecolor='black', color='steelblue')

        # Dotted Lines for Statistics (RESTORED)
        ax1.axvline(x=avg_mag, color='red', linestyle='--', linewidth=2, label=f'Average ({avg_mag:.2f} mm)')
        ax1.axvline(x=avg_minus_2std, color='green', linestyle=':', linewidth=2, label=f'-2 Sigma ({avg_minus_2std:.2f} mm)')
        ax1.axvline(x=avg_plus_2std, color='green', linestyle=':', linewidth=2, label=f'+2 Sigma ({avg_plus_2std:.2f} mm)')
        ax1.axvline(x=avg_minus_3std, color='orange', linestyle=':', linewidth=2, label=f'-3 Sigma ({avg_minus_3std:.2f} mm)')
        ax1.axvline(x=avg_plus_3std, color='orange', linestyle=':', linewidth=2, label=f'+3 Sigma ({avg_plus_3std:.2f} mm)')

        ax1.set_xlabel('Magnitude / Range (mm)', fontsize=12)
        ax1.set_ylabel('Occurrences (Cycles)', fontsize=12)

        filter_text = f"Filtered ≥ {min_mag} mm"
        if filter_outliers:
            filter_text += f" | Outliers > {max_mag} mm removed"

        ax1.set_title(f'Fatigue Cycle Distribution ({filter_text})', fontsize=14, pad=20)
        ax1.grid(axis='y', linestyle='--', alpha=0.7)
        ax1.legend(loc='upper right')

        # Auto Scale the X-Axis (RESTORED)
        plot_min_x = max(0, avg_minus_4std)
        plot_max_x = min(avg_plus_4std, max_mag_val + 0.1)
        ax1.set_xlim(plot_min_x, plot_max_x)

        step = max(0.02, round((plot_max_x - plot_min_x) / 20, 2))
        ax1.set_xticks(np.arange(plot_min_x, plot_max_x + step, step))
        ax1.tick_params(axis='x', rotation=45)

        # Secondary Top Axis for Force (RESTORED)
        ax_hist_force = ax1.twiny()
        ax_hist_force.set_xlim(plot_min_x * stiffness, plot_max_x * stiffness)
        ax_hist_force.set_xlabel(f'Calculated Force (N)  [Static Stiffness: {stiffness:.1f} N/mm]', fontsize=12, color='darkred', labelpad=10)
        ax_hist_force.tick_params(axis='x', colors='darkred')

        plt.tight_layout()
        plt.show()

        # --- Display the Statistics Table (RESTORED) ---
        print(f"\n--- Magnitude & Force Statistics ({filter_text}) ---")
        display(HTML(stats_df.to_html(index=False)))

        # ==========================================
        # PLOT 2: TRENDING (DEGRADATION) ANALYSIS
        # ==========================================
        trend_df = pd.DataFrame({'Time_s': cycle_times, 'Magnitude': magnitudes})
        trend_df = trend_df.sort_values('Time_s').reset_index(drop=True)

        window_size = max(10, len(trend_df) // 100)
        trend_df['Mag_MA'] = trend_df['Magnitude'].rolling(window=window_size, min_periods=1).mean()
        trend_df['Stiffness_MA'] = ref_force / trend_df['Mag_MA']

        fig2, ax_trend1 = plt.subplots(figsize=(14, 6))
        ax_trend1.scatter(trend_df['Time_s'], trend_df['Magnitude'], alpha=0.15, color='gray', s=8, label='Raw Cycles')
        line1 = ax_trend1.plot(trend_df['Time_s'], trend_df['Mag_MA'], color='darkblue', linewidth=2, label=f'Mag. Moving Avg ({window_size} cycles)')

        ax_trend1.set_xlabel('Time (Seconds)', fontsize=12)
        ax_trend1.set_ylabel('Magnitude (mm)', fontsize=12, color='darkblue')
        ax_trend1.tick_params(axis='y', labelcolor='darkblue')
        ax_trend1.set_title('Sample Degradation & Stiffness Trend Over Time', fontsize=14, pad=15)
        ax_trend1.grid(True, linestyle='--', alpha=0.5)

        ax_trend2 = ax_trend1.twinx()
        line2 = ax_trend2.plot(trend_df['Time_s'], trend_df['Stiffness_MA'], color='darkred', linewidth=2, linestyle='-', label=f'Est. Stiffness (Assuming {ref_force} N)')
        ax_trend2.set_ylabel('Estimated Stiffness (N/mm)', fontsize=12, color='darkred')
        ax_trend2.tick_params(axis='y', labelcolor='darkred')

        lines = line1 + line2 + [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8)]
        labels = [l.get_label() for l in line1 + line2] + ['Raw Cycles']
        ax_trend1.legend(lines, labels, loc='best')

        plt.tight_layout()
        plt.show()

        # ==========================================
        # NEW: STIFFNESS DEGRADATION PROOF (Linear Regression)
        # ==========================================
        # 1. Calculate the instantaneous stiffness for every single cycle
        trend_df['Stiffness_Raw'] = ref_force / trend_df['Magnitude']

        # 2. Perform Linear Regression: Stiffness vs Time
        slope, intercept, r_value, p_value, std_err = stats.linregress(trend_df['Time_s'], trend_df['Stiffness_Raw'])
        r_squared = r_value**2

        proof_df = pd.DataFrame({
            'Statistical Metric': ['Degradation Rate (Slope)', 'Significance (p-value)', 'Goodness of Fit (R²)', 'Standard Error'],
            'Value': [f"{slope:.6f} N/mm/s", f"{p_value:.4e}", f"{r_squared:.4f}", f"{std_err:.6f}"]
        })

        print("\n--- STIFFNESS DEGRADATION PROOF ---")
        display(HTML(proof_df.to_html(index=False)))

        # 3. Final Verification Statement
        if p_value < 0.05:
            display(HTML(f"<div style='background-color: #d4edda; padding: 10px; border-radius: 5px; color: #155724; font-weight: bold;'>"
                         f"✅ PROOF: A statistically significant degradation trend was detected (p = {p_value:.4e})."
                         f"</div>"))
        else:
            display(HTML(f"<div style='background-color: #f8d7da; padding: 10px; border-radius: 5px; color: #721c24; font-weight: bold;'>"
                         f"❌ NOTICE: No statistically significant degradation detected (p = {p_value:.3f})."
                         f"</div>"))

        print("\nAnalysis complete!")

# Link button click to the function
process_button.on_click(analyze_fatigue_data)